In [1]:
import logging
import random
import time
import uuid
from datetime import datetime, UTC

from confluent_kafka import SerializingProducer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import StringSerializer
from faker import Faker

In [2]:
def generate_product(fake_: Faker) -> dict:
    return {
        "product_id": str(uuid.uuid4()),
        "name": fake_.catch_phrase(),
        "category": random.choice(["전자기기", "생활용품", "식품", "도서", "의류", "스포츠", "뷰티"]),
        "price": random.randint(1_000, 500_000),
        "currency": "KRW",
        "stock": random.randint(0, 500),
        "created_at": datetime.now(UTC).isoformat()
    }

In [3]:
def delivery_report(err, msg):
    if err is not None:
        logging.info(f"Delivery failed: {err}")
    else:
        logging.info(f"Delivered to {msg.topic()} [{msg.partition()}] @ offset {msg.offset()}")

In [5]:
_value_schema_str = """
{
  "type": "record",
  "name": "Product",
  "namespace": "com.example.product",
  "fields": [
    {"name": "product_id", "type": "string"},
    {"name": "name", "type": "string"},
    {"name": "category", "type": "string"},
    {"name": "price", "type": "int"},
    {"name": "currency", "type": "string", "default": "KRW"},
    {"name": "stock", "type": "int"},
    {"name": "created_at", "type": "string"}
  ]
}
"""

_schema_registry_client = SchemaRegistryClient({"url": "http://confluent-schema-registry:8081"})
_producer_avro_conf = {
    "bootstrap.servers": "confluent-kafka-broker:9093",
    "security.protocol": "SASL_PLAINTEXT",
    "sasl.mechanism": "PLAIN",
    "sasl.username": "admin",
    "sasl.password": "admin",
    "key.serializer": StringSerializer("utf_8"),
    "value.serializer": AvroSerializer(schema_registry_client=_schema_registry_client, schema_str=_value_schema_str, to_dict=lambda obj, ctx: obj),
}

_producer_avro_zstd_conf = {
    "bootstrap.servers": "confluent-kafka-broker:9093",
    "security.protocol": "SASL_PLAINTEXT",
    "sasl.mechanism": "PLAIN",
    "sasl.username": "admin",
    "sasl.password": "admin",
    "key.serializer": StringSerializer("utf_8"),
    "value.serializer": AvroSerializer(schema_registry_client=_schema_registry_client, schema_str=_value_schema_str, to_dict=lambda obj, ctx: obj),
    "compression.type": "zstd"
}

In [6]:
_schema_registry_client.get_subjects()
#_schema_registry_client.delete_subject("mmix-products-topic-avro-value")

[]

In [7]:
_fake = Faker("ko_KR")
_topic_avro = "mmix-products-topic-avro"
_producer_avro = SerializingProducer(_producer_avro_conf)
_topic_avro_zstd = "mmix-products-topic-avro-zstd"
_producer_avro_zstd = SerializingProducer(_producer_avro_zstd_conf)

In [8]:
for i in range(10):
    product = generate_product(_fake)
    _producer_avro.produce(topic=_topic_avro, key=product["product_id"], value=product, on_delivery=delivery_report)
    _producer_avro_zstd.produce(topic=_topic_avro_zstd, key=product["product_id"], value=product, on_delivery=delivery_report)
    time.sleep(0.1)

_producer_avro.flush(10)
_producer_avro_zstd.flush(10)

0